In [ ]:
!pip install -q streamlit crewai yfinance pandas numpy groq langchain-openai pyngrok


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
!pip install litellm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 29.1 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0


In [ ]:
%%writefile app.py
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import os
from crewai import Agent, Task, Crew, Process, LLM

# ==========================================
# 1. STREAMLIT UI SETUP
# ==========================================
st.set_page_config(page_title="AI Investment Committee", layout="wide")
st.title("🤖 Multi-Agent AI Investment Committee Dashboard")
st.caption("Powered by CrewAI, Groq (Llama 3.3), and Financial Engineering Mathematics")

# Sidebar for configuration inputs
with st.sidebar:
    st.header("Configuration")
    groq_api_key = st.text_input("Enter Groq API Key:", type="password")
    ticker = st.selectbox("Select Stock Ticker:", ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "SAP", "SIE.DE"])
    run_button = st.button("Run Committee Analysis", type="primary")

# ==========================================
# 2. FINANCIAL DATA FETCHING ENGINE (FREE)
# ==========================================
def fetch_financial_data(ticker_symbol):
    stock = yf.Ticker(ticker_symbol)
    df = stock.history(period="1y")

    if df.empty:
        return None

    # --- MATH ENGINE: Technical Indicators ---
    df['SMA_50'] = df['Close'].rolling(window=50).mean()

    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-10)
    df['RSI_14'] = 100 - (100 / (1 + rs))

    # --- MATH ENGINE: Risk Metrics ---
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    daily_volatility = df['Log_Returns'].std()
    annualized_volatility = daily_volatility * np.sqrt(252)

    latest = df.iloc[-1]

    metrics = {
        "current_price": round(latest['Close'], 2),
        "sma_50": round(latest['SMA_50'], 2) if not pd.isna(latest['SMA_50']) else "N/A",
        "rsi_14": round(latest['RSI_14'], 2) if not pd.isna(latest['RSI_14']) else "N/A",
        "annualized_volatility": round(annualized_volatility * 100, 2) if not pd.isna(annualized_volatility) else "N/A"
    }
    return metrics

# ==========================================
# 3. MAIN RUNTIME LOGIC
# ==========================================
if run_button:
    if not groq_api_key:
        st.error("Please provide your Groq API key in the sidebar to proceed.")
    else:
        with st.spinner(f"Extracting market indicators and assembling the AI committee for {ticker}..."):
            metrics = fetch_financial_data(ticker)

            if metrics is None:
                st.error("Failed to retrieve data for the chosen ticker.")
            else:
                col1, col2, col3, col4 = st.columns(4)
                col1.metric("Current Price", f"${metrics['current_price']}")
                col2.metric("50-Day SMA", f"${metrics['sma_50']}")
                col3.metric("14-Day RSI", f"{metrics['rsi_14']}")
                col4.metric("Annualized Volatility", f"{metrics['annualized_volatility']}%")

                # FIX 1 & 2: Injection of user input key and fixed code indentation block
                os.environ["GROQ_API_KEY"] = groq_api_key
                llm = LLM(
                    model="groq/llama-3.3-70b-versatile",
                    api_key=groq_api_key
                )

                quant_analyst = Agent(
                    role="Technical Quant Analyst",
                    goal=f"Analyze the raw price metrics for {ticker} and determine macro market momentum.",
                    backstory="You are a strict data-driven mathematical trader. You look only at price deviations, SMA crossovers, and RSI momentum conditions.",
                    llm=llm,
                    verbose=True
                )

                risk_manager = Agent(
                    role="Risk Manager",
                    goal=f"Determine optimal allocation caps for {ticker} to preserve institutional fund safety.",
                    backstory="You are a conservative risk officer. You use asset volatility metrics to enforce strict capital preservation rules and prevent catastrophic losses.",
                    llm=llm,
                    verbose=True
                )

                committee_chair = Agent(
                    role="Investment Committee Chair",
                    goal="Synthesize technical momentum and risk insights into a definitive investment directive.",
                    backstory="You are the ultimate decision-maker. You evaluate arguments objectively and deliver a finalized Buy, Sell, or Hold rating with explicit analytical rationale.",
                    llm=llm,
                    verbose=True
                )

                task1 = Task(
                    description=f"Evaluate these raw numbers for {ticker}: Price={metrics['current_price']}, 50-day SMA={metrics['sma_50']}, 14-day RSI={metrics['rsi_14']}. Write a quantitative report checking if the stock is overbought (RSI > 70), oversold (RSI < 30), or trending bullishly above its SMA.",
                    expected_output="A structured paragraph assessing momentum direction and a clear technical momentum score from 1-10.",
                    agent=quant_analyst
                )

                task2 = Task(
                    description=f"Analyze the annualized historical asset volatility calculated for {ticker}: {metrics['annualized_volatility']}%. Based on standard risk practices, recommend a maximum allocation percentage of a $1,000,000 portfolio to this asset.",
                    expected_output="A risk statement detailing capital allocation limits and a calculated risk rating from 1-10.",
                    agent=risk_manager
                )

                task3 = Task(
                    description="Review the Technical Quant Analyst's report and the Risk Manager's capital advice. Create a final executive investment advisory brief.",
                    expected_output="A complete, clean Markdown brief containing three clear sections: 1. Final Action Verdict (BUY/SELL/HOLD), 2. Synthesis of Technical and Risk logic, 3. Allocation and Execution Strategy.",
                    agent=committee_chair
                )

                crew = Crew(
                    agents=[quant_analyst, risk_manager, committee_chair],
                    tasks=[task1, task2, task3],
                    process=Process.sequential
                )

                result = crew.kickoff()

                st.success("Analysis Complete!")
                st.markdown("### 📋 Final Committee Briefing Document")
                st.markdown(result.raw)


Overwriting app.py


In [ ]:
!wget -qO- https://localedit.com || curl ifconfig.me


34.125.191.183

In [ ]:
# 1. Automated installation guard to prevent ModuleNotFoundError
try:
    from pyngrok import ngrok
except ModuleNotFoundError:
    print("📦 Installing missing tunnel dependencies...")
    import os
    os.system("pip install -q pyngrok")
    from pyngrok import ngrok

import time

# 2. Clear out any old running backend instances instantly
!pkill -f streamlit
!pkill -f ngrok

# 3. Authenticate your tunnel agent
AUTHTOKEN = "x"
ngrok.set_auth_token(AUTHTOKEN)

# 4. Launch Streamlit via Colab's raw system runner
print("🚀 Launching Streamlit server...")
get_ipython().system_raw("python -m streamlit run app.py --server.port 8501 --server.address 127.0.0.1 &")
time.sleep(4)  # Short pause to let it initialize cleanly

# 5. Connect the tunnel interface safely
print("🔗 Creating secure public access tunnel...")
try:
    ngrok.disconnect_all() # Clean any ghost tunnels
except:
    pass

public_url = ngrok.connect(8501, bind_tls=True)

# 6. Print out the actionable dashboard links immediately
print("\n" + "="*50)
print("🔥 DASHBOARD IS ONLINE!")
print(f"Click here to open: {public_url.public_url}")
print("="*50)


🚀 Launching Streamlit server...
🔗 Creating secure public access tunnel...

🔥 DASHBOARD IS ONLINE!
Click here to open: https://thinly-captive-humorless.ngrok-free.dev


In [ ]:
import os
import time

print("🧹 1. Sweeping old processes and cleaning environment...")
!pkill -f streamlit
!pkill -f ngrok

print("📦 2. Verifying and installing high-end quantitative libraries...")
!pip install -q streamlit crewai yfinance pandas numpy groq langchain-openai pyngrok litellm arch --upgrade

# --- REWRITING THE PERFECT STREAMLIT APP FILE WITH THE ARRAY EXTRACTION FIX ---
print("✍️ 3. Generating a high-fidelity app.py script...")
with open("app.py", "w") as f:
    f.write('''import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import os
from crewai import Agent, Task, Crew, Process, LLM
from arch import arch_model

# Framework patch for Groq prompt-caching bug
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

st.set_page_config(page_title="AI Investment Committee", layout="wide")
st.title("🤖 Multi-Agent AI Investment Committee Dashboard")
st.caption("Production Architecture: CrewAI, Groq Llama 3.3, GARCH(1,1), Historical VaR, and Live Fundamental Parsing")

with st.sidebar:
    st.header("Configuration")
    groq_api_key = st.text_input("Enter Groq API Key:", type="password")
    ticker = st.selectbox("Select Stock Ticker:", ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "SAP", "SIE.DE"])
    run_button = st.button("Run Committee Analysis", type="primary")

def fetch_advanced_financial_data(ticker_symbol):
    stock = yf.Ticker(ticker_symbol)
    df = stock.history(period="1y")
    if df.empty:
        return None

    # --- MATH ENGINE 1: Technical Indicators (RSI & MACD) ---
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-10)
    df['RSI_14'] = 100 - (100 / (1 + rs))

    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # --- MATH ENGINE 2: Log Returns & Value-at-Risk (VaR) ---
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    clean_returns = df['Log_Returns'].dropna()

    # 95% 1-Day Historical Value-at-Risk percentage
    var_95 = -np.percentile(clean_returns, 5) * 100

    # --- MATH ENGINE 3: GARCH(1,1) Volatility Modeling ---
    scaled_returns = clean_returns * 100
    try:
        model_garch = arch_model(scaled_returns, vol='Garch', p=1, q=1, dist='Normal')
        res_garch = model_garch.fit(disp='off')

        forecast_df = res_garch.forecast(horizon=1).variance
        latest_variance = forecast_df.iloc[-1].values

        # FIX: Flatten array and pull item directly to prevent type conversion errors
        garch_annualized_vol = float(np.sqrt(latest_variance).flatten()[0]) * np.sqrt(252)
    except Exception as e:
        garch_annualized_vol = float(clean_returns.std() * np.sqrt(252) * 100)

    # --- TEXT ENGINE: Live Market & Fundamental News Scraping ---
    raw_news = stock.news
    news_summary = ""
    if raw_news:
        for idx, item in enumerate(raw_news[:3]):
            content = item.get('content', {})
            title = content.get('title', 'No Title')
            summary = content.get('summary', '')
            news_summary += f"[{idx+1}] {title}: {summary}\\n"
    else:
        news_summary = "No recent fundamental news transcripts or releases found."

    latest = df.iloc[-1]
    return {
        "current_price": round(latest['Close'], 2),
        "rsi_14": round(latest['RSI_14'], 2) if not pd.isna(latest['RSI_14']) else "N/A",
        "macd": round(latest['MACD'], 3) if not pd.isna(latest['MACD']) else "N/A",
        "macd_signal": round(latest['MACD_Signal'], 3) if not pd.isna(latest['MACD_Signal']) else "N/A",
        "var_95": round(var_95, 2),
        "garch_volatility": round(garch_annualized_vol, 2),
        "news_feed": news_summary
    }

if run_button:
    if not groq_api_key:
        st.error("Please provide your Groq API key in the sidebar to proceed.")
    else:
        with st.spinner(f"Extracting math metrics, parsing filings, and convening advanced investment committee..."):
            metrics = fetch_advanced_financial_data(ticker)
            if metrics is None:
                st.error("Failed to retrieve data for the chosen ticker.")
            else:
                col1, col2, col3, col4 = st.columns(4)
                col1.metric("Current Price", f"${metrics['current_price']}")
                col2.metric("GARCH (1,1) Volatility", f"{metrics['garch_volatility']}%")
                col3.metric("14-Day RSI / MACD", f"{metrics['rsi_14']} / {metrics['macd']}")
                col4.metric("95% Historical VaR", f"{metrics['var_95']}%")

                os.environ["GROQ_API_KEY"] = groq_api_key
                llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=groq_api_key)

                quant_analyst = Agent(
                    role="Technical Quant Analyst",
                    goal=f"Analyze raw technical price structures, MACD trend lines, and GARCH statistical volatility variables for {ticker}.",
                    backstory="You are a quantitative specialist. You interpret momentum oscillators and conditional heteroskedasticity data frames to identify market regime shifts.",
                    llm=llm,
                    verbose=True
                )

                fundamental_analyst = Agent(
                    role="Fundamental Text Analyst",
                    goal=f"Scan and digest public corporate text streams, news feeds, and narrative data regarding {ticker} to establish sentiment context.",
                    backstory="You are an equity analyst specializing in reading corporate releases, evaluating competitive moats, and finding systemic risks within text transcripts.",
                    llm=llm,
                    verbose=True
                )

                risk_manager = Agent(
                    role="Risk Manager",
                    goal=f"Evaluate Value-at-Risk limits and portfolio exposures for {ticker} to calculate safe asset sizing restrictions.",
                    backstory="You are a conservative risk controller. You translate empirical statistical losses like VaR down into operational capital allocation safety limits.",
                    llm=llm,
                    verbose=True
                )

                task1 = Task(
                    description=(
                        f"Evaluate these data strings for {ticker}: Price={metrics['current_price']}, "
                        f"RSI={metrics['rsi_14']}, MACD={metrics['macd']} (Signal Line={metrics['macd_signal']}), "
                        f"and GARCH Annualized Volatility={metrics['garch_volatility']}%. "
                        "Determine if technical momentum is strengthening or decaying based on crossovers and variance levels."
                    ),
                    expected_output="A quantitative market momentum assessment paragraph and an absolute momentum score (1-10).",
                    agent=quant_analyst
                )

                task2 = Task(
                    description=f"Read and evaluate the following text metrics extracted from recent market releases for {ticker}:\\n{metrics['news_feed']}\\nIdentify core operational highlights, corporate sentiment indices, and potential business challenges mentioned.",
                    expected_output="A corporate fundamental briefing analyzing business health and a sentiment bias indicator (Bullish/Bearish/Neutral).",
                    agent=fundamental_analyst
                )

                task3 = Task(
                    description=(
                        f"Review the momentum report from the Quant and the corporate narrative from the Fundamental analyst. "
                        f"Factor in that {ticker} possesses a 95% 1-day Historical Value-at-Risk (VaR) of {metrics['var_95']}%, "
                        "meaning there is a 5% historical probability of losing that specific percentage of asset value in a single day. "
                        "Synthesize these three domains into an executive corporate mandate."
                    ),
                    expected_output="A comprehensive Markdown memo containing: 1. Action Verdict (BUY/SELL/HOLD), 2. Mathematical Rationale (referencing GARCH and VaR explicitly), 3. Allocation and Capital Protection Blueprint.",
                    agent=risk_manager
                )

                crew = Crew(
                    agents=[quant_analyst, fundamental_analyst, risk_manager],
                    tasks=[task1, task2, task3],
                    process=Process.sequential
                )

                result = crew.kickoff()
                st.success("Analysis Complete!")
                st.markdown(result.raw)
''')

print("🚀 4. Launching background Streamlit host...")
get_ipython().system_raw("python -m streamlit run app.py --server.port 8501 --server.address 127.0.0.1 &")
time.sleep(5)

print("🔗 5. Building Ngrok secure pipeline tunnel...")
from pyngrok import ngrok
AUTHTOKEN = "-"
ngrok.set_auth_token(AUTHTOKEN)
try:
    ngrok.disconnect_all()
except:
    pass

public_url = ngrok.connect(8501, bind_tls=True)

print(f"\n🔥 FULL REBOOT SUCCESSFUL!\nClick here to launch your app: {public_url.public_url}")


🧹 1. Sweeping old processes and cleaning environment...


📦 2. Verifying and installing high-end quantitative libraries...
✍️ 3. Generating a high-fidelity app.py script...
🚀 4. Launching background Streamlit host...
🔗 5. Building Ngrok secure pipeline tunnel...

🔥 FULL REBOOT SUCCESSFUL!
Click here to launch your app: https://thinly-captive-humorless.ngrok-free.dev
